In [7]:
import pandas as pd

In [8]:
# Load the data
df = pd.read_csv('../data/raw/upi_2024.csv')

In [14]:
# Initial data exploration
print("=== DATA OVERVIEW ===")
print("First 5 rows:")
print(df.head())


=== DATA OVERVIEW ===
First 5 rows:
  transaction_id            timestamp transaction_type merchant_category  \
0  TXN0000000001  2024-10-08 15:17:28              P2P     Entertainment   
1  TXN0000000002  2024-04-11 06:56:00              P2M           Grocery   
2  TXN0000000003  2024-04-02 13:27:18              P2P           Grocery   
3  TXN0000000004  2024-01-07 10:09:17              P2P              Fuel   
4  TXN0000000005  2024-01-23 19:04:23              P2P          Shopping   

   amount transaction_status sender_age_group receiver_age_group  \
0     868            SUCCESS            26-35              18-25   
1    1011            SUCCESS            26-35              26-35   
2     477            SUCCESS            26-35              36-45   
3    2784            SUCCESS            26-35              26-35   
4     990            SUCCESS            26-35              18-25   

    sender_state sender_bank receiver_bank device_type network_type  \
0          Delhi        Axi

In [15]:
print("Dataset info:")
print(df.info())


Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 17 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   transaction_id      250000 non-null  object
 1   timestamp           250000 non-null  object
 2   transaction_type    250000 non-null  object
 3   merchant_category   250000 non-null  object
 4   amount              250000 non-null  int64 
 5   transaction_status  250000 non-null  object
 6   sender_age_group    250000 non-null  object
 7   receiver_age_group  250000 non-null  object
 8   sender_state        250000 non-null  object
 9   sender_bank         250000 non-null  object
 10  receiver_bank       250000 non-null  object
 11  device_type         250000 non-null  object
 12  network_type        250000 non-null  object
 13  fraud_flag          250000 non-null  int64 
 14  hour_of_day         250000 non-null  int64 
 15  day_of_week         250000 non-null  

In [16]:
print("Basic statistics:")
print(df.describe())

Basic statistics:
              amount     fraud_flag    hour_of_day     is_weekend
count  250000.000000  250000.000000  250000.000000  250000.000000
mean     1311.756036       0.001920      14.681032       0.285348
std      1848.059224       0.043776       5.188304       0.451581
min        10.000000       0.000000       0.000000       0.000000
25%       288.000000       0.000000      11.000000       0.000000
50%       629.000000       0.000000      15.000000       0.000000
75%      1596.000000       0.000000      19.000000       1.000000
max     42099.000000       1.000000      23.000000       1.000000


In [19]:
# Data cleaning
print("\n=== DATA CLEANING ===")
# Convert timestamp
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f"Timestamp data type: {type(df['timestamp'][0])}")


=== DATA CLEANING ===
Timestamp data type: <class 'pandas._libs.tslibs.timestamps.Timestamp'>


In [20]:
# Clean column names
df.columns = [i.strip().lower().replace(' ','_') for i in df.columns]
print(f"Cleaned column names: {df.columns.tolist()}")

Cleaned column names: ['transaction_id', 'timestamp', 'transaction_type', 'merchant_category', 'amount', 'transaction_status', 'sender_age_group', 'receiver_age_group', 'sender_state', 'sender_bank', 'receiver_bank', 'device_type', 'network_type', 'fraud_flag', 'hour_of_day', 'day_of_week', 'is_weekend']


In [21]:
# Rename amount column
df.rename(columns={'amount_(inr)': 'amount'}, inplace=True)
print("Renamed 'amount_(inr)' to 'amount'")

Renamed 'amount_(inr)' to 'amount'


In [29]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

Missing values:
transaction_id        0
timestamp             0
transaction_type      0
merchant_category     0
amount                0
transaction_status    0
sender_age_group      0
receiver_age_group    0
sender_state          0
sender_bank           0
receiver_bank         0
device_type           0
network_type          0
fraud_flag            0
hour_of_day           0
day_of_week           0
is_weekend            0
dtype: int64


In [30]:
# Data analysis
print("=== DATA ANALYSIS ===")
# Unique values analysis
print("Unique values in each column:")
print(df.nunique())


=== DATA ANALYSIS ===
Unique values in each column:
transaction_id        250000
timestamp             248610
transaction_type           4
merchant_category         10
amount                 10355
transaction_status         2
sender_age_group           5
receiver_age_group         5
sender_state              10
sender_bank                8
receiver_bank              8
device_type                3
network_type               4
fraud_flag                 2
hour_of_day               24
day_of_week                7
is_weekend                 2
dtype: int64


In [31]:
# Device types
print(f"\nUnique device types: {df['device_type'].unique()}")


Unique device types: ['Android' 'iOS' 'Web']


In [32]:
# Fraud analysis by age group
print("\nFraud cases by sender age group:")
fraud_count = df[df['fraud_flag'] == 1].groupby('sender_age_group').size()
print(fraud_count)


Fraud cases by sender age group:
sender_age_group
18-25    143
26-35    163
36-45    116
46-55     31
56+       27
dtype: int64


In [33]:
# Average spending by merchant category
print("\nAverage spending by merchant category (highest to lowest):")
average_spent = df.groupby('merchant_category')['amount'].mean().sort_values(ascending=False)
print(average_spent)



Average spending by merchant category (highest to lowest):
merchant_category
Education        5094.017636
Shopping         2573.085398
Utilities        2361.110305
Fuel             1555.383434
Grocery          1166.350979
Other             848.777550
Healthcare        542.853905
Food              531.694480
Entertainment     413.325374
Transport         308.003780
Name: amount, dtype: float64


In [34]:
# Pivot table analysis
print("\nAverage amount by merchant category and age group:")
pivot_result = df.pivot_table(values='amount', 
                             index='merchant_category', 
                             columns='sender_age_group', 
                             aggfunc='mean')
print(pivot_result)



Average amount by merchant category and age group:
sender_age_group         18-25        26-35        36-45        46-55  \
merchant_category                                                       
Education          3097.595645  5287.210607  6716.587277  5900.342742   
Entertainment       502.329744   440.259847   365.706309   301.153310   
Food                570.195075   560.031250   497.158234   469.533672   
Fuel               1387.726472  1531.052993  1777.435190  1674.260142   
Grocery             951.784301  1177.441471  1341.554724  1253.892979   
Healthcare          332.723813   471.735550   623.931291   778.438003   
Other               920.624717   870.734209   817.673286   756.935444   
Shopping           3034.895198  2741.448533  2312.287671  1948.524446   
Transport           340.700589   308.630527   303.398394   283.086767   
Utilities          1619.251929  2225.286219  2782.168026  3101.591556   

sender_age_group           56+  
merchant_category               
Educa

In [35]:
# Save processed data
df.to_csv('../data/processed/upi_2024.csv', index=False)
print("\nProcessed data saved to '../data/processed/upi_2024.csv'")


Processed data saved to '../data/processed/upi_2024.csv'
